# 05 — GRU: full fixed recipe và ablation class weighting

Notebook 04 đã hoàn tất **pilot**. Notebook này chạy toàn bộ Base.csv:
**640.000 train / 160.000 validation / 200.000 random test**, hai loss
`none` và `balanced`, cùng seed 1234, mỗi loss **8 epoch**.

Dùng lại `src/baf.py` và `src/gru.py`; giữ notebook 01–04 và kết quả cũ.
`Run All` chạy huấn luyện → kiểm tra → xuất proof một lượt, không phụ thuộc
Ctrl+S hoặc tên file notebook. Lưu notebook sau khi hoàn tất để giữ thêm output giao diện.

Nguồn: *Objective over Architecture: Fraud Detection Under Extreme Imbalance
in Bank Account Opening*, DOI `10.3390/computation13120290`.
Commit tác giả: `6e8252cb9080378905b2aae30033c5946b72335b`.
Đây là chạy lại **cấu hình cố định** kèm unweighted ablation, không chạy lại HPO.


## 1. Kernel và source

Có thể dùng kernel CPU hiện tại. Khi dùng venv GPU, chọn kernel của venv đó;
`device="auto"` chỉ chọn CUDA nếu chính kernel này thấy CUDA.
Notebook không cài hoặc thay thư viện. Restart Kernel trước khi Run All.


In [1]:
from pathlib import Path
import json
import math
import sys

PROJECT_DIR = None  # Hoặc r"D:\UIT\DS317\BAF_project"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (path for path in [here, *here.parents] if (path / "src/baf.py").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở BAF_project hoặc điền PROJECT_DIR ở cell này.")

sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf
import gru

for module in [baf, gru]:
    expected_path = (ROOT / "src" / f"{module.__name__}.py").resolve()
    if Path(module.__file__).resolve() != expected_path:
        raise RuntimeError("Kernel đang giữ source khác. Restart Kernel rồi Run All.")

EXPECTED_SOURCE_SHA256 = {
    "baf.py": "0d888da1ce96e33adabb22d44d8fc0ed25da5cf3a4226e862d9dfeac39c3aa12",
    "gru.py": "abdb62ee4b6c6c2241f19dd7e9cd471aeee9fcf95358511d1d2105afb28aa9e6",
}
for filename, digest in EXPECTED_SOURCE_SHA256.items():
    baf.require(
        baf.sha256(ROOT / "src" / filename) == digest,
        f"Source {filename} khác bản pilot đã kiểm chứng. Kiểm tra lại trước khi chạy full.",
    )

print("Project:", ROOT)
print("Kernel:", sys.executable)
print(pd.Series(baf.package_versions(include_torch=True)).to_string())
print("Torch CUDA build:", gru.torch.version.cuda)
print("CUDA available:", gru.torch.cuda.is_available())
if gru.torch.cuda.is_available():
    print("GPU:", gru.torch.cuda.get_device_name(0))


Project: D:\UIT\DS317\BAF_project
Kernel: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
numpy                  2.5.4
pandas                 3.1.0
scipy                 1.18.1
scikit-learn           1.9.1
psutil                 7.2.2
lightgbm               4.7.0
torch           2.14.1+cu130
Torch CUDA build: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 2. Cấu hình full

Giữ cấu hình cố định: embedding 186, hidden 733, Adam lr
0.000448103301026688, batch 256, dropout 0.3. Hai loss có cùng khởi tạo
và thứ tự minibatch theo seed. Không giảm kích thước model/batch nếu thiếu bộ nhớ.

`balanced` tính trọng số từ **train**: `N / (2 * n_c)`.
Đây là một seed để chạy full đầu tiên; chưa đủ kết luận ổn định giữa nhiều seed.


In [2]:
DATA_PATH = ROOT / "data/raw/Base.csv"
EXPECTED_ROWS = 1_000_000
EXPECTED_DATA_SHA256 = (
    "7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809"
)

CONFIG = gru.Config(
    protocol="author_fixed",
    mode="full",
    methods=["none", "balanced"],
    seeds=[1234],
    threads=4,
    device="auto",  # Có thể đặt "cpu" hoặc "cuda" để chọn tường minh.
    embedding_dim=186,
    hidden_dim=733,
    learning_rate=0.000448103301026688,
    batch_size=256,
    epochs=8,
    dropout=0.3,
    save_models=True,
)
CONFIG.validate()

DEVICE = (
    ("cuda" if gru.torch.cuda.is_available() else "cpu")
    if CONFIG.device == "auto"
    else CONFIG.device
)
if DEVICE.startswith("cuda") and not gru.torch.cuda.is_available():
    raise RuntimeError("Kernel này chưa thấy CUDA. Chọn đúng kernel hoặc dùng CPU.")

RUN_DIR = baf.new_directory(ROOT / "results/runs", "gru_author_fixed_full")
print(CONFIG)
print("Device:", DEVICE)
print("Output mới:", RUN_DIR)


Config(protocol='author_fixed', mode='full', methods=['none', 'balanced'], seeds=[1234], threads=4, device='auto', embedding_dim=186, hidden_dim=733, learning_rate=0.000448103301026688, epochs=8, batch_size=256, dropout=0.3, pilot_train=8192, pilot_eval=4096, save_models=True, dataset_tag='BAF_Base')
Device: cuda
Output mới: D:\UIT\DS317\BAF_project\results\runs\gru_author_fixed_full_20261027_123100_deeb80e4


## 3. Kiểm tra dữ liệu và dự trù thời gian

SHA-256 đối chiếu đúng Base.csv đã dùng ở notebook 01–04.
Chia random phân tầng 80/20, rồi chia outer train 80/20: tổng cộng 64/16/20.
Giữ preprocessing fit trên outer 80% như code tác giả, bao gồm feature của inner
validation. Mọi tập đều có thể chứa tháng 7. Không gọi đây là temporal holdout.

Pilot CPU đã đo: 8.192 train, một epoch, `none` 50,05 giây và `balanced`
48,32 giây, bốn thread. Ngoại suy tuyến tính cho hai loss full khoảng **17,1 giờ**,
chưa gồm toàn bộ preprocessing, inference và đóng gói. Đây không phải thời gian full đã đo.
GPU dùng môi trường khác cần số đo riêng.


In [3]:
baf.require(DATA_PATH.is_file(), f"Không thấy dữ liệu: {DATA_PATH}")
baf.require(
    baf.sha256(DATA_PATH) == EXPECTED_DATA_SHA256,
    "Base.csv khác dữ liệu đã chạy pilot. Kiểm tra đường dẫn và nội dung file.",
)
frame = baf.load_data(DATA_PATH, expected_rows=EXPECTED_ROWS)
outer_rows = len(frame) - int(math.ceil(len(frame) * 0.2))
validation_rows = int(math.ceil(outer_rows * 0.2))
expected_parts = {
    "train": outer_rows - validation_rows,
    "validation": validation_rows,
    "random_test": len(frame) - outer_rows,
}
display(pd.Series(expected_parts, name="Expected rows"))
print("Fraud trong toàn bộ dữ liệu:", int(frame.fraud_bool.sum()))

if DEVICE == "cpu":
    pilot_seconds_per_epoch = {"none": 50.0524538, "balanced": 48.3230116}
    estimated_fit_hours = (
        sum(pilot_seconds_per_epoch[method] for method in CONFIG.methods)
        * expected_parts["train"]
        / 8192
        * CONFIG.epochs
        * len(CONFIG.seeds)
        / 3600
    )
    print(f"Ngoại suy thời gian fit CPU: khoảng {estimated_fit_hours:.1f} giờ.")
    print("Mốc tham khảo từ pilot cũ, không phải benchmark của lượt full này.")
else:
    print("Chưa có số đo GPU; không áp dụng ngoại suy từ pilot CPU.")


train          640000
validation     160000
random_test    200000
Name: Expected rows, dtype: int64

Fraud trong toàn bộ dữ liệu: 11029
Chưa có số đo GPU; không áp dụng ngoại suy từ pilot CPU.


## 4. Huấn luyện

Cell này chạy hai loss liên tiếp, mỗi loss 8 epoch. Mỗi dòng `EPOCH` là một
epoch đã xong; trên CPU có thể chờ khoảng một giờ giữa hai dòng.
Loss và thời gian mỗi epoch được ghi ngay vào `epoch_history.csv` và log.
Checkpoint được ghi sau khi hoàn tất từng loss; chưa có resume giữa epoch.
Giữ máy thức và kernel hoạt động trong suốt lượt chạy.


In [4]:
metrics = gru.train(
    data_path=DATA_PATH,
    output=RUN_DIR,
    config=CONFIG,
    frame=frame,
    expected_rows=EXPECTED_ROWS,
)
del frame
display(metrics)


D:\UIT\DS317\BAF_project\src\gru.py:292: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(part.fraud_bool.to_numpy(dtype=np.int64)),


{
  "protocol": "author_fixed",
  "mode": "full",
  "methods": [
    "none",
    "balanced"
  ],
  "seeds": [
    1234
  ],
  "threads": 4,
  "device": "auto",
  "embedding_dim": 186,
  "hidden_dim": 733,
  "learning_rate": 0.000448103301026688,
  "epochs": 8,
  "batch_size": 256,
  "dropout": 0.3,
  "pilot_train": 8192,
  "pilot_eval": 4096,
  "save_models": true,
  "dataset_tag": "BAF_Base",
  "evaluation_partitions": [
    "validation",
    "random_test"
  ]
}
                  n positive                    months
train        640000     7058  [0, 1, 2, 3, 4, 5, 6, 7]
validation   160000     1765  [0, 1, 2, 3, 4, 5, 6, 7]
random_test  200000     2206  [0, 1, 2, 3, 4, 5, 6, 7]
Author fixed recipe: random64/16/20, preprocessing fit outer80% as code. Inner validation features enter preprocessing. No HPO. Seed reset and paired unweighted ablation added. Random split includes month7.
EPOCH {"method": "none", "seed": 1234, "epoch": 1, "loss_batch_mean": 0.050815234273066744, "seconds": 88

,protocol,mode,method,seed,partition,threshold,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,initial_model_sha256,...,precision_frozen_threshold,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp
0,author_fixed,full,gru_none,1234,validation,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.101273,0.054988,0.027762,0.690141,0.984155,0.989138,150328,7907,874,891
1,author_fixed,full,gru_none,1234,random_test,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.103279,0.055045,0.024479,0.692308,0.984081,0.989120,187922,9872,1069,1137
2,author_fixed,full,gru_balanced,1234,validation,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.101003,0.054825,0.822096,0.039559,0.864960,0.777863,150349,7886,879,886
3,author_fixed,full,gru_balanced,1234,random_test,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.104269,0.054810,0.828196,0.039941,0.865382,0.778530,187975,9819,1063,1143


## 5. Kiểm tra và đọc kết quả

Audit đọc lại **bốn file predictions**, tính lại metric, kiểm tra hai checkpoint,
đủ 16 dòng epoch history và cùng khởi tạo giữa hai loss.

- Bảng ngưỡng 0,5 gần cách báo cáo fixed recipe hơn; vẫn cần đối chiếu cùng metric,
  split và seed trước khi nói tái lập được điểm paper.
- Ngưỡng FPR 5% được chọn trên validation rồi **giữ nguyên** khi đánh giá random test.
  FPR trên random test có thể vượt 5%.
- `recall_at_fpr5_curve` của random test là phân tích đường ROC dùng nhãn test,
  không dùng để chọn ngưỡng vận hành hoặc chọn cấu hình.

Không ghép điểm này vào leaderboard temporal của notebook 03. GRU dùng chuỗi
các cột của từng hồ sơ, không phải chuỗi giao dịch theo thời gian.


In [5]:
report = baf.audit_results([RUN_DIR], reference="gru_none")
manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
history = pd.read_csv(RUN_DIR / "epoch_history.csv")

baf.require(manifest["mode"] == "full", "Kết quả này chưa phải mode full.")
for name, count in expected_parts.items():
    baf.require(manifest["partitions"][name]["n"] == count, f"Sai split: {name}")
baf.require(len(report.prediction_audit) == 4, "Phải có bốn file prediction.")
baf.require(len(history) == 16, "Phải đủ hai loss, mỗi loss tám epoch.")

display(pd.DataFrame(manifest["partitions"]).T)
display(history)
display(
    report.metrics[
        [
            "method",
            "seed",
            "partition",
            "ap",
            "roc_auc",
            "recall_at_0p5",
            "precision_at_0p5",
            "f1_weighted_at_0p5",
            "accuracy_at_0p5",
        ]
    ]
)
display(
    report.metrics[
        [
            "method",
            "partition",
            "threshold",
            "recall_frozen_threshold",
            "fpr_frozen_threshold",
            "precision_frozen_threshold",
            "tn",
            "fp",
            "fn",
            "tp",
        ]
    ]
)
display(report.deltas)
fit_cost = report.metrics.drop_duplicates(["method", "seed"])
display(
    fit_cost[
        [
            "method",
            "seed",
            "fit_seconds",
            "peak_rss_mib",
            "peak_cuda_allocated_mib",
            "weight_negative",
            "weight_positive",
            "warnings",
        ]
    ]
)
print(f"Tổng thời gian fit đã đo: {fit_cost.fit_seconds.sum() / 3600:.2f} giờ.")
print("FULL_GRU_READY:", RUN_DIR.name)


,n,positive,months
train,640000,7058,"[0, 1, 2, 3, 4, 5, 6, 7]"
validation,160000,1765,"[0, 1, 2, 3, 4, 5, 6, 7]"
random_test,200000,2206,"[0, 1, 2, 3, 4, 5, 6, 7]"


,method,seed,epoch,loss_batch_mean,seconds
0,none,1234,1,0.050815,88.446356
1,none,1234,2,0.048632,85.370888
2,none,1234,3,0.047169,83.512641
3,none,1234,4,0.046492,84.452716
4,none,1234,5,0.046296,83.610522
5,none,1234,6,0.045891,83.143754
6,none,1234,7,0.045557,85.130376
7,none,1234,8,0.045165,83.222519
8,balanced,1234,1,0.495411,85.270322
9,balanced,1234,2,0.474703,84.767845


,method,seed,partition,ap,roc_auc,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5
0,gru_none,1234,validation,0.153392,0.882407,0.027762,0.690141,0.984155,0.989138
1,gru_none,1234,random_test,0.162971,0.887598,0.024479,0.692308,0.984081,0.989120
2,gru_balanced,1234,validation,0.156373,0.876728,0.822096,0.039559,0.864960,0.777863
3,gru_balanced,1234,random_test,0.156023,0.883154,0.828196,0.039941,0.865382,0.778530


,method,partition,threshold,recall_frozen_threshold,fpr_frozen_threshold,precision_frozen_threshold,tn,fp,fn,tp
0,gru_none,validation,0.059153,0.504816,0.049970,0.101273,150328,7907,874,891
1,gru_none,random_test,0.059153,0.515413,0.049911,0.103279,187922,9872,1069,1137
2,gru_balanced,validation,0.851337,0.501983,0.049837,0.101003,150349,7886,879,886
3,gru_balanced,random_test,0.851337,0.518132,0.049643,0.104269,187975,9819,1063,1143


,reference,method,seed,partition,delta_recall_pp,delta_ap
0,gru_none,gru_balanced,1234,validation,-0.283286,0.002981
1,gru_none,gru_balanced,1234,random_test,0.317316,-0.006948


,method,seed,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,weight_negative,weight_positive,warnings
0,gru_none,1234,676.940642,1929.324219,405.203125,1.000000,1.000000,NaN
2,gru_balanced,1234,673.456177,1940.730469,405.025391,0.505576,45.338623,NaN


Tổng thời gian fit đã đo: 0.38 giờ.
FULL_GRU_READY: gru_author_fixed_full_20261027_123100_deeb80e4


## 6. Xuất proof một lần

Giữ source, manifest/config, split indices, preprocessing, log, epoch history,
checkpoints, predictions, metric và SHA-256. ZIP chứa `verify.ipynb` có output
kiểm chứng đọc lại thực tế; không phải bản sao giao diện notebook chưa lưu.
Không huấn luyện lại khi xuất proof, không cần Ctrl+S trước cell này.


In [6]:
archive = baf.export_proof(
    run_paths=[RUN_DIR],
    output_root=ROOT / "results/proofs",
    reference="gru_none",
)
print("Hoàn tất full GRU và proof:", archive)


Verified prediction files: 4
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_125434_633b6f23.zip
Hoàn tất full GRU và proof: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_125434_633b6f23.zip
